# Algorithm 26: PPR-Only Lattice Projection

This notebook tests the cleaner Algorithm26 version:

```text
KLDM-PC state -> predict clean lattice -> project clean lattice -> renoise lattice at same t -> continue / inspect
```

No CPS state update is used in Algorithm26. The only intervention is lattice PPR:

```text
ell_t^PPR = mean_t(ell0_projected_soft) + sigma_t * eps_mix
```

Fractional coordinates and KLDM velocity are preserved for lattice-only PPR diagnostics.


In [ ]:
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")
os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")

import importlib
from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace
from typing import Any

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

ROOT = Path.cwd().resolve()
if not ((ROOT / 'configs').exists() and (ROOT / 'src').exists()):
    if ((ROOT.parent / 'configs').exists() and (ROOT.parent / 'src').exists()):
        ROOT = ROOT.parent

import kldmPlus.algorithm25_kldm_pc_cps_lattice as kspace_backend
import kldmPlus.algorithm25_raw_native_lattice_cps as raw_backend
import kldmPlus.algorithm26_lattice_ppr_rescue as algo26_backend
kspace_backend = importlib.reload(kspace_backend)
raw_backend = importlib.reload(raw_backend)
algo26_backend = importlib.reload(algo26_backend)

from kldmPlus.run_sampling_compare import SamplingCompareRunner, set_seed
from kldmPlus.sample_evaluation import build_structure_from_sample, evaluate_csp_reconstruction as evaluate_csp_reconstruction_plus
from kldm.sample_evaluation import evaluate_csp_reconstruction as evaluate_csp_reconstruction_kldm

CONFIG_PATH = ROOT / 'configs' / 'kldm_plus' / 'mp_20' / 'mp20_sampling_compare_em_vs_algorithm10_casal_chart.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

SAMPLE_SEED = int(os.environ.get('ALGO26_SEED', '2'))
GRAPH_IDS = [int(v.strip()) for v in os.environ.get('ALGO26_GRAPH_IDS', '1,2,3').split(',') if v.strip()]
ALGO26_TOTAL_STEPS = int(os.environ.get('ALGO26_TOTAL_STEPS', '800'))
ALGO26_REMAINING_STEPS = [int(v.strip()) for v in os.environ.get('ALGO26_REMAINING_STEPS', '400,200').split(',') if v.strip()]
ALGO26_PROJECTION_MODES = [v.strip() for v in os.environ.get('ALGO26_PROJECTION_MODES', 'none,kspace,raw_native').split(',') if v.strip()]
ALGO26_RHO_LATTICE_VALUES = [float(v.strip()) for v in os.environ.get('ALGO26_RHO_L_VALUES', '1.0,0.75,0.5').split(',') if v.strip()]
ALGO26_RUN_FULL_SAMPLER = os.environ.get('ALGO26_RUN_FULL_SAMPLER', '0').strip().lower() in {'1','true','yes'}
ALGO26_DEBUG_SAMPLER = os.environ.get('ALGO26_DEBUG_SAMPLER', '1').strip().lower() in {'1','true','yes'}
ALGO26_DEBUG_EVERY = int(os.environ.get('ALGO26_DEBUG_EVERY', '100'))

runner = SamplingCompareRunner(config_path=CONFIG_PATH)
runner.model.eval()
set_seed(SAMPLE_SEED)

requested_num_targets = max(max(GRAPH_IDS) if GRAPH_IDS else 0, len(GRAPH_IDS), 5)
if int(runner.compare_cfg.get('num_targets', 0)) < requested_num_targets:
    runner.compare_cfg['num_targets'] = int(requested_num_targets)
    runner.compare_cfg['batch_size'] = max(int(runner.compare_cfg.get('batch_size', 0)), int(requested_num_targets))
    runner.loader, runner.lattice_transform = runner._build_loader()

full_batch = next(iter(runner.loader)).to(runner.device)
full_ptr = full_batch.ptr.tolist()
full_num_graphs = max(len(full_ptr) - 1, 0)
available_graph_ids = [int(graph_id) for graph_id in GRAPH_IDS if 1 <= int(graph_id) <= full_num_graphs]
selected_graph_indices0 = [int(graph_id) - 1 for graph_id in available_graph_ids]
selected_items = full_batch.index_select(selected_graph_indices0) if hasattr(full_batch, 'index_select') else full_batch[selected_graph_indices0]
if isinstance(selected_items, list):
    batch = full_batch.__class__.from_data_list(selected_items)
else:
    batch = selected_items
batch = batch.to(runner.device)
ptr = batch.ptr.detach().cpu().tolist()

ALGO26_CACHE: dict[tuple[Any, ...], Any] = {}
result_tables: dict[str, pd.DataFrame] = {}

print(f'algo26_ppr_only graphs={available_graph_ids} seed={SAMPLE_SEED} total_steps={ALGO26_TOTAL_STEPS} remaining_steps={ALGO26_REMAINING_STEPS} modes={ALGO26_PROJECTION_MODES} rhos={ALGO26_RHO_LATTICE_VALUES}')


In [ ]:
@dataclass
class GraphCase:
    graph_id: int
    graph_idx0: int
    atomic_numbers: torch.Tensor
    gt_frac: torch.Tensor
    gt_l_feature: torch.Tensor
    requested_sg: int


def dataframe_result(name: str, rows: list[dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if 'PASS' not in df.columns:
        df['PASS'] = False
    if 'status' not in df.columns:
        df['status'] = np.where(df['PASS'], 'PASS', 'FAIL')
    result_tables[name] = df
    return df


def safe_display_sorted(df: pd.DataFrame, by: list[str]):
    df = df.copy()
    for key in by:
        if key not in df.columns:
            df[key] = np.nan
    display(df.sort_values(by).reset_index(drop=True))


def safe_metric_float(value) -> float:
    if value is None:
        return float('nan')
    if torch.is_tensor(value):
        value = value.detach().reshape(-1)
        if value.numel() == 0:
            return float('nan')
        value = value[0].item()
    try:
        return float(value)
    except Exception:
        return float('nan')


def safe_metric_bool(value) -> bool:
    if torch.is_tensor(value):
        value = value.detach().reshape(-1)
        if value.numel() == 0:
            return False
        value = value[0].item()
    return bool(value)


def _cache_key(*parts: Any) -> tuple[Any, ...]:
    return tuple(parts)


def _cached(key: tuple[Any, ...], fn):
    if key not in ALGO26_CACHE:
        ALGO26_CACHE[key] = fn()
    return ALGO26_CACHE[key]


def graph_slice(graph_idx0: int) -> tuple[int, int]:
    return int(ptr[graph_idx0]), int(ptr[graph_idx0 + 1])


def graph_tensors(graph_idx0: int) -> dict[str, Any]:
    start, end = graph_slice(graph_idx0)
    return {
        'pos': batch.pos[start:end].detach().clone(),
        'l': batch.l[graph_idx0].detach().clone().reshape(-1),
        'h': batch.atomic_numbers[start:end].detach().clone().to(torch.long),
        'sg': int(torch.as_tensor(batch.space_group).reshape(-1)[graph_idx0].item()),
    }


def load_test_graphs(graph_ids=available_graph_ids) -> list[GraphCase]:
    out = []
    for graph_idx0, graph_id in enumerate(graph_ids):
        g = graph_tensors(graph_idx0)
        out.append(GraphCase(
            graph_id=int(graph_id),
            graph_idx0=int(graph_idx0),
            atomic_numbers=g['h'],
            gt_frac=g['pos'],
            gt_l_feature=g['l'],
            requested_sg=g['sg'],
        ))
    return out


GRAPH_CASES = load_test_graphs(available_graph_ids)
print('loaded_graphs=', [g.graph_id for g in GRAPH_CASES], 'sg=', [g.requested_sg for g in GRAPH_CASES])


def graph_edge_node_index(case: GraphCase) -> torch.Tensor:
    start, end = graph_slice(case.graph_idx0)
    edge_index = batch.edge_node_index
    mask = ((edge_index[0] >= start) & (edge_index[0] < end) & (edge_index[1] >= start) & (edge_index[1] < end))
    return (edge_index[:, mask] - start).detach().clone()


def make_single_graph_batch_view(case: GraphCase, *, ref_tensor: torch.Tensor | None = None):
    device = case.gt_frac.device if ref_tensor is None else ref_tensor.device
    dtype = case.gt_frac.dtype if ref_tensor is None else ref_tensor.dtype
    pos = case.gt_frac.detach().clone().to(device=device, dtype=dtype)
    l = case.gt_l_feature.detach().clone().reshape(1, -1).to(device=device, dtype=case.gt_l_feature.dtype)
    atomic_numbers = case.atomic_numbers.detach().clone().to(device=device, dtype=torch.long)
    batch_index = torch.zeros(pos.shape[0], device=device, dtype=torch.long)
    num_atoms = torch.tensor([int(pos.shape[0])], device=device, dtype=torch.long)
    edge_node_index = graph_edge_node_index(case).to(device=device)
    space_group = torch.tensor([int(case.requested_sg)], device=device, dtype=torch.long)

    class _SingleGraphBatch(SimpleNamespace):
        def to(self, device):
            out = _SingleGraphBatch()
            for key, value in self.__dict__.items():
                setattr(out, key, value.to(device) if torch.is_tensor(value) else value)
            return out

    return _SingleGraphBatch(
        pos=pos,
        l=l,
        atomic_numbers=atomic_numbers,
        batch=batch_index,
        num_atoms=num_atoms,
        num_graphs=1,
        edge_node_index=edge_node_index,
        space_group=space_group,
    )


def remaining_step_to_tau(remaining_step: int) -> float:
    return float(remaining_step) / float(max(ALGO26_TOTAL_STEPS, 1))


In [ ]:
def cell_lengths_angles(cell: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    cell = torch.as_tensor(cell).reshape(3, 3)
    lengths = torch.linalg.norm(cell, dim=1)
    def angle(u, v):
        denom = (torch.linalg.norm(u) * torch.linalg.norm(v)).clamp_min(1.0e-12)
        cosine = torch.clamp(torch.dot(u, v) / denom, -1.0, 1.0)
        return torch.rad2deg(torch.acos(cosine))
    return lengths, torch.stack([angle(cell[1], cell[2]), angle(cell[0], cell[2]), angle(cell[0], cell[1])])


def cell_volume(cell: torch.Tensor) -> float:
    return float(torch.abs(torch.det(torch.as_tensor(cell).reshape(3, 3))).detach().item())


def lattice_metric_dict(pred_l: torch.Tensor, target_l: torch.Tensor, case: GraphCase) -> dict[str, float]:
    num_atoms = int(case.atomic_numbers.shape[0])
    pred_cell = kspace_backend.lattice6_to_matrix(pred_l, num_atoms=num_atoms, lattice_transform=runner.lattice_transform)
    gt_cell = kspace_backend.lattice6_to_matrix(target_l, num_atoms=num_atoms, lattice_transform=runner.lattice_transform)
    pred_lengths, pred_angles = cell_lengths_angles(pred_cell)
    gt_lengths, gt_angles = cell_lengths_angles(gt_cell)
    pred_volume = cell_volume(pred_cell)
    gt_volume = cell_volume(gt_cell)
    return {
        'cell_matrix_rmse': float(torch.sqrt(torch.mean((pred_cell - gt_cell) ** 2)).detach().item()),
        'length_rmse': float(torch.sqrt(torch.mean((pred_lengths - gt_lengths) ** 2)).detach().item()),
        'angle_rmse_deg': float(torch.sqrt(torch.mean((pred_angles - gt_angles) ** 2)).detach().item()),
        'volume_abs_error': abs(pred_volume - gt_volume),
        'volume_rel_error': abs(pred_volume - gt_volume) / max(abs(gt_volume), 1.0e-12),
    }


def evaluate_arrays(case: GraphCase, pred_f: torch.Tensor, pred_l: torch.Tensor, pred_h: torch.Tensor) -> dict[str, Any]:
    plus = evaluate_csp_reconstruction_plus(
        pred_f=pred_f,
        pred_l=pred_l,
        pred_a=pred_h.to(torch.long),
        target_f=case.gt_frac,
        target_l=case.gt_l_feature,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
        requested_space_group=case.requested_sg,
        validity_cutoff=0.5,
    )
    kldm = evaluate_csp_reconstruction_kldm(
        pred_f=pred_f,
        pred_l=pred_l,
        pred_a=pred_h.to(torch.long),
        target_f=case.gt_frac,
        target_l=case.gt_l_feature,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
    )
    return {'match': bool(kldm.match), 'valid': bool(kldm.valid), 'rmse': kldm.rmse, 'frac_rmse': plus.frac_rmse, 'plus_match': bool(plus.match), 'plus_valid': bool(plus.valid), 'plus_rmse': plus.rmse}


def detect_spacegroup_family(pred_f: torch.Tensor, pred_l: torch.Tensor, pred_a: torch.Tensor) -> dict[str, Any]:
    try:
        structure = build_structure_from_sample(pred_f, pred_l, pred_a, lattice_transform=runner.lattice_transform)
        sg = int(SpacegroupAnalyzer(structure, symprec=1.0e-1, angle_tolerance=5.0).get_space_group_number())
        return {'detected_space_group': sg, 'detected_family': kspace_backend.spacegroup_to_crystal_family(sg)}
    except Exception:
        return {'detected_space_group': None, 'detected_family': None}


def projection_config(mode: str, *, rho: float = 0.75) -> algo26_backend.Algorithm26PPRConfig:
    return algo26_backend.Algorithm26PPRConfig(
        total_steps=int(ALGO26_TOTAL_STEPS),
        projection_mode=mode,
        projection_start_frac=0.25,
        projection_interval=50,
        rho_lattice=float(rho),
        gamma_min=0.05,
        gamma_max=3.0,
        gamma_power=2.0,
        preserve_lattice_volume=False,
        raw_angle_scale_deg=10.0,
        raw_shift_cap_scaled=1.0,
        raw_use_gate=False,
        induced_small_angstrom=0.05,
        induced_large_angstrom=0.25,
    )


In [ ]:
def local_noisy_state(case: GraphCase, *, remaining_step: int, seed: int = SAMPLE_SEED) -> dict[str, Any]:
    tau = remaining_step_to_tau(remaining_step)
    batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
    set_seed(int(seed) + 1000 * int(case.graph_id) + int(remaining_step))
    t_graph = torch.as_tensor([[tau]], device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    t_nodes = torch.full((int(case.gt_frac.shape[0]),), tau, device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    f_t, v_t, *_ = runner.model.tdm.sample_noisy_state(t=t_nodes, f0=case.gt_frac, index=batch_view.batch)
    t_lattice = t_graph.reshape(-1)
    set_seed(int(seed) + 2000 * int(case.graph_id) + int(remaining_step))
    l_t, _ = runner.model.diffusion_l.forward_sample(
        t=t_lattice,
        x0=case.gt_l_feature.reshape(1, -1),
        num_atoms=batch_view.num_atoms,
    )
    return {
        'f_t': f_t,
        'v_t': v_t,
        'l_t': l_t.reshape(1, -1),
        'a_t': case.atomic_numbers,
        'node_index': batch_view.batch,
        'edge_node_index': batch_view.edge_node_index,
        't_graph': t_graph,
        't_nodes': t_nodes,
        't_lattice': t_lattice,
        'batch_view': batch_view,
        'tau': tau,
    }


def clean_prediction(case: GraphCase, state: dict[str, Any]) -> algo26_backend.Algorithm26CleanPrediction:
    return algo26_backend.predict_clean_lattice_and_fractional(
        model=runner.model,
        f_t=state['f_t'],
        v_t=state['v_t'],
        l_t=state['l_t'],
        atom_types=state['a_t'],
        node_index=state['node_index'],
        edge_node_index=state['edge_node_index'],
        t_graph=state['t_graph'],
        t_nodes=state['t_nodes'],
        t_lattice=state['t_lattice'],
        num_atoms=state['batch_view'].num_atoms,
    )


def project_for_case(case: GraphCase, clean: algo26_backend.Algorithm26CleanPrediction, *, mode: str, tau: float, rho: float = 0.75) -> algo26_backend.Algorithm26Projection:
    return algo26_backend.project_clean_lattice_for_ppr(
        ell0_hat=clean.ell0_hat,
        f0_hat=clean.f0_hat,
        num_atoms=int(case.atomic_numbers.shape[0]),
        lattice_transform=runner.lattice_transform,
        space_group_number=int(case.requested_sg),
        tau=float(tau),
        config=projection_config(mode, rho=rho),
    )


def clean_after_lattice(case: GraphCase, state: dict[str, Any], l_new: torch.Tensor) -> algo26_backend.Algorithm26CleanPrediction:
    state_after = dict(state)
    state_after['l_t'] = l_new.reshape(1, -1)
    return clean_prediction(case, state_after)


### Test A: Lattice Roundtrip And GT Projection Identity

This checks the geometry plumbing before PPR:

- KLDM lattice feature -> matrix -> feature roundtrip
- KLDM feature -> DiffCSP++ k -> feature roundtrip
- GT projected through raw-native and k-space family projection

If GT identity fails for a mode, that mode is basis-sensitive for that graph.

In [ ]:
rows_a = []
for case in GRAPH_CASES:
    num_atoms = int(case.atomic_numbers.shape[0])
    ell = case.gt_l_feature.reshape(-1)
    cell = kspace_backend.lattice6_to_matrix(ell, num_atoms=num_atoms, lattice_transform=runner.lattice_transform)
    ell_matrix_round = kspace_backend.matrix_to_lattice6(cell, num_atoms=num_atoms, lattice_transform=runner.lattice_transform)
    k = kspace_backend.lattice6_to_k(ell, num_atoms=num_atoms, lattice_transform=runner.lattice_transform)
    ell_k_round = kspace_backend.k_to_lattice6(k, num_atoms=num_atoms, lattice_transform=runner.lattice_transform)
    k_proj = kspace_backend.project_k_to_spacegroup_family(k, space_group_number=int(case.requested_sg), preserve_lattice_volume=False)
    ell_k_proj = kspace_backend.k_to_lattice6(k_proj, num_atoms=num_atoms, lattice_transform=runner.lattice_transform)
    ell_raw_proj, raw_diag = raw_backend.raw_native_project_lattice6(
        ell,
        num_atoms=num_atoms,
        lattice_transform=runner.lattice_transform,
        space_group_number=int(case.requested_sg),
        angle_scale_deg=10.0,
        shift_cap_scaled=1.0,
    )
    for source, pred in [
        ('matrix_roundtrip', ell_matrix_round),
        ('k_roundtrip', ell_k_round),
        ('kspace_gt_projection', ell_k_proj),
        ('raw_native_gt_projection', ell_raw_proj),
    ]:
        metrics = lattice_metric_dict(pred, ell, case)
        rows_a.append({
            'test': 'algorithm26_a_roundtrip_gt_identity',
            'graph': case.graph_id,
            'space_group': int(case.requested_sg),
            'family': kspace_backend.spacegroup_to_crystal_family(int(case.requested_sg)),
            'source': source,
            **metrics,
            'k_violation_before': safe_metric_float(kspace_backend.k_family_violation(k, space_group_number=int(case.requested_sg))),
            'k_violation_after': safe_metric_float(kspace_backend.k_family_violation(k_proj, space_group_number=int(case.requested_sg))) if source == 'kspace_gt_projection' else float('nan'),
            'raw_selected_variant': raw_diag.selected_variant if source == 'raw_native_gt_projection' else '',
            'PASS': bool(metrics['length_rmse'] < 1e-4 and metrics['angle_rmse_deg'] < 1e-3) if source in {'matrix_roundtrip','k_roundtrip'} else True,
            'status': 'INFO',
        })

safe_display_sorted(dataframe_result('algorithm26_a_roundtrip_gt_identity', rows_a), ['graph', 'source'])


### Test B: PPR Identity Reningoise

This is the most important implementation sanity test.

If `ell0_projected == ell0_hat` and `rho_lattice = 1`, PPR should reconstruct the original noisy lattice state exactly:

```text
ell_t = mean_t(ell0_hat) + sigma_t * eps_old
```

If this fails, the lattice marginal / MatterGen prior handling is wrong.

In [ ]:
rows_b = []
for case in GRAPH_CASES:
    for remaining_step in ALGO26_REMAINING_STEPS:
        state = local_noisy_state(case, remaining_step=int(remaining_step))
        clean = clean_prediction(case, state)
        ident = algo26_backend.lattice_ppr_renoise_from_clean(
            ell_t=state['l_t'],
            ell0_hat=clean.ell0_hat,
            ell0_projected=clean.ell0_hat,
            t_lattice=state['t_lattice'],
            diffusion_l=runner.model.diffusion_l,
            num_atoms=state['batch_view'].num_atoms,
            rho_lattice=1.0,
            noise=torch.zeros_like(state['l_t']),
        )
        max_abs = float(torch.max(torch.abs(ident.l_t.reshape(-1) - state['l_t'].reshape(-1))).detach().item())
        rows_b.append({
            'test': 'algorithm26_b_ppr_identity_renoise',
            'graph': case.graph_id,
            'remaining_step': int(remaining_step),
            'tau': remaining_step_to_tau(int(remaining_step)),
            'max_abs_lattice_reconstruction_error': max_abs,
            'lattice_state_shift_norm': safe_metric_float(ident.lattice_state_shift_norm),
            'PASS': bool(max_abs < 1e-5),
            'status': 'INFO',
        })

safe_display_sorted(dataframe_result('algorithm26_b_ppr_identity_renoise', rows_b), ['graph', 'remaining_step'])


### Test C: Projection Feasibility At Different t

At selected noise levels, we predict `ell0_hat`, project it, and measure whether the clean projection is a small/medium/large basis move.

This does not run the sampler; it is a cheap feasibility map.

In [ ]:
rows_c = []
for case in GRAPH_CASES:
    for remaining_step in ALGO26_REMAINING_STEPS:
        state = local_noisy_state(case, remaining_step=int(remaining_step))
        clean = clean_prediction(case, state)
        clean_metrics = lattice_metric_dict(clean.ell0_hat, case.gt_l_feature, case)
        for mode in ALGO26_PROJECTION_MODES:
            proj = project_for_case(case, clean, mode=mode, tau=state['tau'], rho=0.75)
            hard_metrics = lattice_metric_dict(proj.ell0_projected, case.gt_l_feature, case)
            soft_metrics = lattice_metric_dict(proj.ell0_soft, case.gt_l_feature, case)
            rows_c.append({
                'test': 'algorithm26_c_projection_feasibility_by_t',
                'graph': case.graph_id,
                'remaining_step': int(remaining_step),
                'tau': state['tau'],
                'space_group': int(case.requested_sg),
                'projection_mode': mode,
                'family': proj.family,
                'zone': proj.zone,
                'weight': safe_metric_float(proj.weight),
                'clean_lattice_rmse_before': clean_metrics['cell_matrix_rmse'],
                'hard_projected_lattice_rmse': hard_metrics['cell_matrix_rmse'],
                'soft_projected_lattice_rmse': soft_metrics['cell_matrix_rmse'],
                'induced_cart_shift_rms': safe_metric_float(proj.induced_cart_shift_rms),
                'relative_cell_shift': safe_metric_float(proj.relative_cell_shift),
                'max_length_change_percent': safe_metric_float(proj.max_length_change_percent),
                'max_angle_change_deg': safe_metric_float(proj.max_angle_change_deg),
                'violation_before': safe_metric_float(proj.violation_before),
                'violation_after': safe_metric_float(proj.violation_after),
                'PASS': bool(proj.accepted),
                'status': 'INFO',
            })

safe_display_sorted(dataframe_result('algorithm26_c_projection_feasibility_by_t', rows_c), ['graph', 'remaining_step', 'projection_mode'])


### Test D: PPR Rho Sweep At Different t

Now we actually renoise the projected clean lattice back to the same noise level.

We preserve `F_t,V_t` and only replace `ell_t`. Then we ask KLDM for the next clean lattice estimate again. This tests whether PPR gives the denoiser a better lattice state than the original noisy state.

The controls are:

```text
baseline
lattice_ppr_no_projection
lattice_ppr_kspace
lattice_ppr_raw_native
```

This separates the effect of renoising from the effect of projection.

In [ ]:
rows_d = []
for case in GRAPH_CASES:
    for remaining_step in ALGO26_REMAINING_STEPS:
        state = local_noisy_state(case, remaining_step=int(remaining_step))
        clean = clean_prediction(case, state)
        base_metrics = lattice_metric_dict(clean.ell0_hat, case.gt_l_feature, case)
        rows_d.append({
            'test': 'algorithm26_d_ppr_rho_sweep_by_t',
            'graph': case.graph_id,
            'remaining_step': int(remaining_step),
            'tau': state['tau'],
            'method': 'baseline',
            'projection_mode': 'baseline',
            'zone': 'none',
            'rho_lattice': float('nan'),
            'projection_weight': 0.0,
            'lattice_state_shift_norm': 0.0,
            'projected_mean_shift_norm': 0.0,
            'clean_lattice_rmse_before': base_metrics['cell_matrix_rmse'],
            'clean_lattice_rmse_after_ppr': base_metrics['cell_matrix_rmse'],
            'noisy_lattice_rmse_after_ppr': lattice_metric_dict(state['l_t'].reshape(-1), case.gt_l_feature, case)['cell_matrix_rmse'],
            'length_rmse_after_ppr': base_metrics['length_rmse'],
            'angle_rmse_after_ppr_deg': base_metrics['angle_rmse_deg'],
            'ppr_improves_clean_lattice': False,
            'PASS': True,
            'status': 'INFO',
        })
        for mode in ALGO26_PROJECTION_MODES:
            proj = project_for_case(case, clean, mode=mode, tau=state['tau'], rho=0.75)
            for rho in ALGO26_RHO_LATTICE_VALUES:
                set_seed(SAMPLE_SEED + int(case.graph_id) * 10000 + int(remaining_step) * 10 + int(rho * 100))
                ppr = algo26_backend.apply_lattice_ppr_projection(
                    ell_t=state['l_t'],
                    t_lattice=state['t_lattice'],
                    diffusion_l=runner.model.diffusion_l,
                    num_atoms=state['batch_view'].num_atoms,
                    projection=proj,
                    rho_lattice=float(rho),
                    use_soft_projected_clean=True,
                )
                clean_after = clean_after_lattice(case, state, ppr.l_t)
                after_metrics = lattice_metric_dict(clean_after.ell0_hat, case.gt_l_feature, case)
                noisy_metrics = lattice_metric_dict(ppr.l_t.reshape(-1), case.gt_l_feature, case)
                rows_d.append({
                    'test': 'algorithm26_d_ppr_rho_sweep_by_t',
                    'graph': case.graph_id,
                    'remaining_step': int(remaining_step),
                    'tau': state['tau'],
                    'method': f"lattice_ppr_{'no_projection' if mode in {'none','no_projection'} else mode}",
                    'projection_mode': mode,
                    'zone': proj.zone,
                    'rho_lattice': float(rho),
                    'projection_weight': safe_metric_float(proj.weight),
                    'lattice_state_shift_norm': safe_metric_float(ppr.lattice_state_shift_norm),
                    'projected_mean_shift_norm': safe_metric_float(ppr.projected_mean_shift_norm),
                    'clean_lattice_rmse_before': base_metrics['cell_matrix_rmse'],
                    'clean_lattice_rmse_after_ppr': after_metrics['cell_matrix_rmse'],
                    'noisy_lattice_rmse_after_ppr': noisy_metrics['cell_matrix_rmse'],
                    'length_rmse_after_ppr': after_metrics['length_rmse'],
                    'angle_rmse_after_ppr_deg': after_metrics['angle_rmse_deg'],
                    'ppr_improves_clean_lattice': bool(after_metrics['cell_matrix_rmse'] <= base_metrics['cell_matrix_rmse']),
                    'PASS': True,
                    'status': 'INFO',
                })

safe_display_sorted(dataframe_result('algorithm26_d_ppr_rho_sweep_by_t', rows_d), ['graph', 'remaining_step', 'method', 'rho_lattice'])


### Test E: Optional Full PC800 PPR-Only Sampler

Disabled by default because this is the expensive part. Enable with:

```bash
ALGO26_RUN_FULL_SAMPLER=1
```

This compares facitKLDM PC against Algorithm26 PPR-only PC for the current default config.

In [ ]:
rows_e = []
if ALGO26_RUN_FULL_SAMPLER:
    for case in GRAPH_CASES[:1]:
        batch_view = make_single_graph_batch_view(case)
        set_seed(SAMPLE_SEED)
        print(f'[algo26-notebook] baseline pc start graph={case.graph_id} steps={ALGO26_TOTAL_STEPS}', flush=True)
        base_f, _base_v, base_l, base_a = runner.model.sample_CSP_algorithm4(
            batch=batch_view,
            n_steps=int(ALGO26_TOTAL_STEPS),
            tau=0.25,
            n_correction_steps=1,
            t_start=1.0,
            t_final=1.0e-6,
        )
        base_eval = evaluate_arrays(case, base_f, base_l.reshape(-1), base_a)
        base_lat = lattice_metric_dict(base_l.reshape(-1), case.gt_l_feature, case)
        base_sym = detect_spacegroup_family(base_f, base_l.reshape(-1), base_a)
        rows_e.append({
            'test': 'algorithm26_e_full_pc_sampler_optional',
            'graph': case.graph_id,
            'method': f'baseline_pc{ALGO26_TOTAL_STEPS}',
            'projection_mode': '',
            'rho_lattice': float('nan'),
            'rmse': safe_metric_float(base_eval.get('rmse')),
            'frac_rmse': safe_metric_float(base_eval.get('frac_rmse')),
            'match': safe_metric_bool(base_eval.get('match')),
            'valid': safe_metric_bool(base_eval.get('valid')),
            'lattice_rmse': base_lat['cell_matrix_rmse'],
            'length_rmse': base_lat['length_rmse'],
            'angle_rmse_deg': base_lat['angle_rmse_deg'],
            'actual_space_group': int(case.requested_sg),
            'detected_space_group': base_sym['detected_space_group'],
            'detected_family': base_sym['detected_family'],
            'num_interventions': 0,
            'PASS': True,
            'status': 'INFO',
        })
        for mode in ['none', 'kspace', 'raw_native']:
            for rho in [0.75]:
                cfg = projection_config(mode, rho=rho)
                print(f'[algo26-notebook] ppr pc start graph={case.graph_id} mode={mode} rho={rho} steps={ALGO26_TOTAL_STEPS}', flush=True)
                out = algo26_backend.kldm_pc_ppr_lattice_sampler(
                    model=runner.model,
                    batch=batch_view,
                    lattice_transform=runner.lattice_transform,
                    oracle_space_group=int(case.requested_sg),
                    config=cfg,
                    debug_label=f'g{case.graph_id}_{mode}_rho{rho}' if ALGO26_DEBUG_SAMPLER else None,
                    debug_print_every=ALGO26_DEBUG_EVERY,
                )
                ev = evaluate_arrays(case, out.frac_coords, out.lattice.reshape(-1), out.atom_types)
                lat = lattice_metric_dict(out.lattice.reshape(-1), case.gt_l_feature, case)
                sym = detect_spacegroup_family(out.frac_coords, out.lattice.reshape(-1), out.atom_types)
                rows_e.append({
                    'test': 'algorithm26_e_full_pc_sampler_optional',
                    'graph': case.graph_id,
                    'method': f"lattice_ppr_{'no_projection' if mode in {'none','no_projection'} else mode}",
                    'projection_mode': mode,
                    'rho_lattice': float(rho),
                    'rmse': safe_metric_float(ev.get('rmse')),
                    'frac_rmse': safe_metric_float(ev.get('frac_rmse')),
                    'match': safe_metric_bool(ev.get('match')),
                    'valid': safe_metric_bool(ev.get('valid')),
                    'lattice_rmse': lat['cell_matrix_rmse'],
                    'length_rmse': lat['length_rmse'],
                    'angle_rmse_deg': lat['angle_rmse_deg'],
                    'actual_space_group': int(case.requested_sg),
                    'detected_space_group': sym['detected_space_group'],
                    'detected_family': sym['detected_family'],
                    'num_interventions': int(len(out.interventions)),
                    'PASS': True,
                    'status': 'INFO',
                })
else:
    print('Full sampler disabled. Set ALGO26_RUN_FULL_SAMPLER=1 to enable.', flush=True)

safe_display_sorted(dataframe_result('algorithm26_e_full_pc_sampler_optional', rows_e), ['graph', 'method', 'projection_mode', 'rho_lattice'])


### Verdict

This cell summarizes whether PPR-only is locally useful before paying for full sampler runs.

In [ ]:
rows_v = []
a = result_tables.get('algorithm26_a_roundtrip_gt_identity', pd.DataFrame())
b = result_tables.get('algorithm26_b_ppr_identity_renoise', pd.DataFrame())
c = result_tables.get('algorithm26_c_projection_feasibility_by_t', pd.DataFrame())
d = result_tables.get('algorithm26_d_ppr_rho_sweep_by_t', pd.DataFrame())
roundtrip_ok = bool(len(a) and a[a['source'].isin(['matrix_roundtrip','k_roundtrip'])]['PASS'].all())
ppr_identity_ok = bool(len(b) and b['PASS'].all())
any_projection_medium_or_small = bool(len(c) and c['zone'].isin(['small','medium']).any())
ppr_improves_some = bool(len(d) and d['ppr_improves_clean_lattice'].any())
if roundtrip_ok and ppr_identity_ok and ppr_improves_some:
    recommendation = 'ppr_only_lattice_projection_is_locally_promising'
elif roundtrip_ok and ppr_identity_ok:
    recommendation = 'ppr_formula_ok_but_projection_not_helpful_yet'
else:
    recommendation = 'fix_lattice_or_ppr_roundtrip_before_sampler_claim'
rows_v.append({
    'test': 'algorithm26_verdict',
    'roundtrip_ok': roundtrip_ok,
    'ppr_identity_ok': ppr_identity_ok,
    'has_small_or_medium_projection': any_projection_medium_or_small,
    'ppr_improves_some_clean_lattice': ppr_improves_some,
    'recommendation': recommendation,
    'PASS': bool(roundtrip_ok and ppr_identity_ok),
    'status': 'INFO',
})
safe_display_sorted(dataframe_result('algorithm26_verdict', rows_v), ['test'])
